# Import libraries

In [ ]:
import numpy as np
import random
import math
import seaborn as sns
import torch
import gc
from src.smithMethods import *
from src.utils.delaunay2d import *
from src.utils.hyperbolicWrappedGaussian import disc_pt_to_hyperboloid, hyperbolic_sampling, proj
from scipy.spatial.qhull import QhullError
from collections import defaultdict 
import time
from matplotlib.lines import Line2D
import networkx as nx
import json
from pathlib import Path
import pickle 
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
import statistics
import biotite.sequence.phylo as phylo
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning) 
FIG_PATH =  Path("./Figures/Results")


from collections import defaultdict 
import pandas as pd
import matplotlib.pyplot as plt
RESULT_PATH =  Path("./Results")

from src.heuristicSteinerDT import heuristicSteinerDT
from src.embed.distances import *
from src.embed.tree_embedders import *

Is the phcpy2c3.so not suited for this platform?


# Define functions

In [ ]:
def normalize(matrix):
    normalized_matrix = (1/np.max(matrix))*matrix
    return normalized_matrix

In [ ]:
def run_realbio_exp(percentage, planaria_klein):
# This is the percentage of missing data i.e. (100-percentage)% of data are considered
    threshold = int((100-percentage)/100 * planaria_klein.shape[0])

    nmbr_permutation = 10 # number of permutations i.e. of different random sampling of the data


    random_seed = 42 
    random.seed(random_seed)
    np.random.seed(random_seed)

    root_idx = 0


    ######################

    length_dict = {'hs':[], 'rhs':[], 'mst_DT': [], 'mst_Kruskal': [], 'nj':[]}
    time_dict = {'hs':[], 'rhs':[], 'mst_DT': [], 'mst_Kruskal': [], 'nj':[]}

    print('random_seed', random_seed)
    print('percentage', percentage)
    print('nmber of permutations', nmbr_permutation)
    print()

    for permutation in range(nmbr_permutation):
      print('permutation', permutation)

      # We keep the root index 0 at 0, and permute the rest
      permutation_indices = np.random.permutation(np.arange(planaria_klein.shape[0])[1:])
      permutation_indices = np.append(root_idx, permutation_indices)

      points_klein = planaria_klein[permutation_indices, :][:threshold]
      nmbr_terminal=points_klein.shape[0]
      print('number of terminals', nmbr_terminal)


      # HyperSteiner 
      hs_results = heuristicSteinerDT(points_klein, method="SLL", space="Klein", triangMeth="DT", maxgroup=4,
                              nIters=3, convDiff=1e-2, dist2Points=1e-5, precise=False, 
                              extendedResults=True, selection= ("MST", (1, 1))
                              )
      length_dict['hs'].append(hs_results['steinerVal'])
      time_dict['hs'].append(hs_results['methodTime'])
      length_dict['mst_DT'].append(hs_results['mstVal'])
      time_dict['mst_DT'].append(hs_results['mstTime'])

      print('hs_length', hs_results['steinerVal'])
      print('hs_time', hs_results['methodTime'])
      print()

      # Optimized Randomized HyperSteiner
      rhs_results = heuristicSteinerDT(points_klein, method="EXH_GLOB", space="Klein", triangMeth="DT", maxgroup=4,
                              nIters=3, convDiff=1e-1, dist2Points=1e-5, precise=False, 
                              extendedResults=True, selection=("Prob", (0.3, 0.6)),
                              nMaxExpansions=int(1*np.sqrt(nmbr_terminal)), slack=2,  threshold_improvement=0.0, 
                              num_epochs=10000, lr=1e-2, early_stopping=True, patience=100, min_delta=1e-6, 
                                )
      length_dict['rhs'].append(rhs_results['steinerVal'])
      time_dict['rhs'].append(rhs_results['methodTime'])
      length_dict['mst_Kruskal'].append(rhs_results['mstVal'])
      time_dict['mst_Kruskal'].append(rhs_results['mstTime'])

      print('rhs_length', rhs_results['steinerVal'])
      print('rhs_time', rhs_results['methodTime'])
      print()

      print('mst_DT_length', hs_results['mstVal'])
      print('mst_DT_time', hs_results['mstTime'])
      print('mst_Kruskal_length', rhs_results['mstVal'])
      print('mst_Kruskal_time', rhs_results['mstTime'])
      print()

      # Neighbor-Joining
      start_time_nj = time.process_time()
      distance_matrix_points_klein = distance_matrix(points_klein)
      njGraph, nj_networkx = neighbor_joining(points_klein, distance_matrix_points_klein) 
      del distance_matrix_points_klein 
      nj_adjacency_matrix, nj_nodes_ordered = adjacency_matrix(nj_networkx) 

      nj_adjacency_matrix = torch.tensor(nj_adjacency_matrix)
      poincare = klein_to_poincare(points_klein)
      poincare = torch.tensor(poincare)

      nj_steiners = train_steiner_embeddings(
          nj_adjacency_matrix,
          terminals_poincare=poincare,
          num_epochs=10000,
          lr=1)

      combined = torch.cat([poincare, nj_steiners], dim=0)
      combined_klein = poincare_to_klein(combined)
      dist_matrix = distance_matrix(combined_klein, klein_distance)        
      nj_length = (0.5*(dist_matrix*nj_adjacency_matrix)).sum()
      nj_time = time.process_time() - start_time_nj

      # Convert tensor to numpy value and clean up memory
      nj_length_value = nj_length.detach().cpu().numpy().item()
      
      # Clean up tensors to free memory
      del nj_adjacency_matrix, poincare, nj_steiners, combined, combined_klein, dist_matrix, nj_length
      
      # Force garbage collection to free GPU/CPU memory
      if torch.cuda.is_available():
          torch.cuda.empty_cache()
      gc.collect()

      length_dict['nj'].append(nj_length_value)
      time_dict['nj'].append(nj_time)

      print('nj_length', nj_length_value)
      print('nj_time', nj_time)
      print()



      # Save results
      name = 'Results/Real/Planaria_rdmseed'+str(random_seed)+'_percent'+str(percentage)+'_nmbrpermutation'+str(nmbr_permutation) 
      with open(name+'_length_dict.pkl', 'wb') as f:
          pickle.dump(length_dict, f)  
      with open(name+'_time_dict.pkl', 'wb') as f:
          pickle.dump(time_dict, f)

    print('DONE!') 

    for key in ['hs', 'rhs', 'mst_DT', 'mst_Kruskal', 'nj']:
        print(key)
        print('length mean+-std '+str(statistics.mean(np.array(length_dict[key])))+'+-'+str(statistics.stdev(np.array(length_dict[key]))))    
        print('time mean+-std '+str(statistics.mean(np.array(time_dict[key])))+'+-'+str(statistics.stdev(np.array(time_dict[key]))))    
        print()

    #return length_dict, time_dict

def run_realbio_exp_all(planaria_klein): #to run from percentages 98 to 99.9, and for 10 samplings at each percentage
    for i in range(1,11):
        run_realbio_exp(99-i/10, planaria_klein)

# Load data, visualize and run experiments

In [ ]:
SAVE_PATH = Path("./Data")
with open(SAVE_PATH / 'planaria_klein_cleaned.npy', 'rb') as f:
    data = np.load(f)

In [ ]:
    fig = plt.figure(figsize=(5,5), dpi=300)
    ax = fig.add_subplot(111, aspect='equal') 


    ax.set_xlim([-1.1,1.1])
    ax.set_ylim([-1.1,1.1])
    

    num_points = 10000
    ax.scatter(data[:num_points, 0], data[:num_points, 1],
                c='tab:blue', s=5, alpha=0.7)
    
    circ = plt.Circle((0, 0), radius=1, edgecolor='grey', facecolor='None', linewidth=3, alpha=1)
    ax.add_patch(circ)
    
    ax.axis("off")
    
    plt.tight_layout()

    fig.savefig(FIG_PATH / f"aistatsplanaria.png")

    plt.show()

In [ ]:
run_realbio_exp_all(data)

# Load the results and rearrange the data structure

In [ ]:
# Load results
random_seed = 10
nmbr_permutation = 10 # number of permutations
percentages = [99.9, 99.8, 99.7, 99.6, 99.5, 99.4, 99.3, 99.2, 99.1, 99.0,
                98.9, 98.8, 98.7, 98.6, 98.5, 98.4, 98.3, 98.2, 98.1, 98.0]
results = {}
for percentage in percentages:
    
    results[str(percentage)] = {'length':[]}
    name = 'Results/Real/Planaria_rdmseed'+str(random_seed)+'_percent'+str(percentage)+'_nmbrpermutation'+str(nmbr_permutation)+'_'
    with open(name+'length_dict.pkl', 'rb') as f:
        results[str(percentage)]['length'] = pickle.load(f)
    with open(name+'time_dict.pkl', 'rb') as f:
        results[str(percentage)]['time'] = pickle.load(f)   
        

In [ ]:
# Rearanging data structure RED OVER MST
nj_length = {}
hs_length = {}
mst_length = {}
rhs_length = {}

nj_time = {}
hs_time = {}
mst_time = {}
rhs_time = {}

for percentage in percentages:

    length_mst = np.array(results[str(percentage)]['length']['mst_DT'])

    nj_length[str(percentage)] = (1-np.array(results[str(percentage)]['length']['nj'])/length_mst)*100
    hs_length[str(percentage)] = (1-np.array(results[str(percentage)]['length']['hs'])/length_mst)*100
    mst_length[str(percentage)] = (1-np.array(results[str(percentage)]['length']['mst_DT'])/length_mst)*100
    rhs_length[str(percentage)] = (1-np.array(results[str(percentage)]['length']['rhs'])/length_mst)*100

    nj_time[str(percentage)] = np.array(results[str(percentage)]['time']['nj'])
    hs_time[str(percentage)] = np.array(results[str(percentage)]['time']['hs'])
    mst_time[str(percentage)] = np.array(results[str(percentage)]['time']['mst_DT'])
    rhs_time[str(percentage)] = np.array(results[str(percentage)]['time']['rhs'])
    
    

# Plot the results

In [ ]:
#Plot performances

sns.set(style="darkgrid")

x = np.linspace(98, 99.9, 20)

transparency = 0.1
divide_factor = 1.0
linewidth = 1.8

xlabels = ['20', '', '','','100','', '','','200','', '','','','300', '', '','','','400','']
fig, ax = plt.subplots(figsize=(7, 4.2))

y_nj = [nj_length[str(percentage)].mean() for percentage in percentages]
std_nj = np.array([nj_length[str(percentage)].std() for percentage in percentages])
std_nj = std_nj / divide_factor
ax.fill_between(x, y_nj - std_nj, y_nj + std_nj, color='#66800B', alpha=transparency)
ax.plot(x, y_nj, color='#66800B', label='NJ', linestyle='--', dashes=(5, 5), linewidth=linewidth)

y_hs = [hs_length[str(percentage)].mean() for percentage in percentages]
std_hs = np.array([hs_length[str(percentage)].std() for percentage in percentages])
std_hs = std_hs / divide_factor
ax.fill_between(x, y_hs - std_hs, y_hs + std_hs, color='#AF3029', alpha=transparency)
ax.plot(x, y_hs, color='#AF3029', label = 'HS', linestyle='--', dashes=(5, 5), linewidth=linewidth)

y_rhs = [rhs_length[str(percentage)].mean() for percentage in percentages]
std_rhs = np.array([rhs_length[str(percentage)].std() for percentage in percentages])
std_rhs = std_rhs / divide_factor
ax.fill_between(x, y_rhs - std_rhs, y_rhs + std_rhs, color='#D0A215', alpha=transparency)
ax.plot(x, y_rhs, color='#D0A215', label = 'RHS', linewidth=linewidth)

# y_mst = [mst_length[str(percentage)].mean() for percentage in percentages]
# std_mst = np.array([mst_length[str(percentage)].std() for percentage in percentages])
# std_mst = std_mst / divide_factor
# ax.fill_between(x, y_mst - std_mst, y_mst + std_mst, color='tab:blue', alpha=transparency)
# ax.plot(x, y_mst, color='tab:blue', label='MST')

ax.set_xlabel(r"$|P|$", fontsize=18)
ax.set_ylabel("RED (%)", fontsize=18)
title = 'Results/Real/results_realdata_red'

plt.xticks(x, xlabels)
#ax.legend(loc='upper center').get_frame().set_facecolor('white')

ax.set_xlim(98, 99.9)
ax.patch.set_alpha(0.5)

ax.tick_params(axis='both', which='major', labelsize=17)

plt.tight_layout()
plt.savefig(title+'.pdf')
plt.savefig(title+'.png')
plt.show()